In [5]:
%pip cache purge
%pip install duckdb polars pandas pyarrow seaborn plotly

Files removed: 0 (0 bytes)
Directories removed: 0
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import duckdb
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    con.close()
except:
    pass

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("""
CREATE OR REPLACE SECRET seaweed_s3 (
    TYPE S3,
    KEY_ID 'anonymous',
    SECRET 'anonymous',
    REGION 'us-east-1',
    ENDPOINT 's3.seaweed.home.arpa',
    URL_STYLE 'path',
    USE_SSL false
);
""")

# 2. Run your query and fetch directly
df_trades = con.sql(trades_sql).df()   # or .pl() if you prefer Polars
df_trades.head(10)

IOException: IO Error: Could not connect to server error for HTTP GET to 'http://s3.seaweed.home.arpa/symm/?encoding-type=url&list-type=2&prefix=virtual%2Fclosed%2F'

In [7]:
trades_sql = """
WITH raw_positions AS (
    SELECT 
        epoch,
        tick,
        to_timestamp(timestamp / 1e9) AS timestamp,
        json_extract_string(position, '$.id') AS id,
        json_extract_string(position, '$.symbol') AS symbol,
        json_extract_string(position, '$.book') AS book,
        json_extract_string(position, '$.entry_path') AS entry_path,
        json_extract_string(position, '$.exit_path') AS exit_path,
        json_extract_string(position, '$.exit_trigger') AS exit_trigger,
        TRY_CAST(json_extract(position, '$.qty') AS DOUBLE) AS qty,
        TRY_CAST(json_extract(position, '$.entry_price') AS DOUBLE) AS entry_price,
        TRY_CAST(json_extract(position, '$.exit_price') AS DOUBLE) AS exit_price,
        TRY_CAST(json_extract(position, '$.pnl_pct') AS DOUBLE) AS pnl_pct,
        TRY_CAST(json_extract(position, '$.realized_pct') AS DOUBLE) AS realized_pct,
        TRY_CAST(json_extract(position, '$.peak_pnl_pct') AS DOUBLE) AS peak_pnl_pct,
        CAST(json_extract(position, '$.hold_ns') AS BIGINT) / 1e9 AS hold_seconds,
        json_extract_string(position, '$.opened_at') AS opened_at,
        json_extract_string(position, '$.closed_at') AS closed_at,
        -- Stoploss A/B Comparison data
        TRY_CAST(json_extract(position, '$.official_exit_pct') AS DOUBLE) AS structural_exit_pct,
        TRY_CAST(json_extract(position, '$.learning.exit_pct') AS DOUBLE) AS learning_exit_pct,
        json_extract_string(position, '$.learning.exit_trigger') AS learning_trigger
    FROM read_json(
        ['s3://symm/virtual/closed/**', 's3://symm/virtual/closed/**'], 
        union_by_name = true
    )
)
SELECT 
    symbol,
    book,
    entry_path,
    exit_trigger,
    round(pnl_pct, 2) AS structural_pnl_pct,
    round(learning_exit_pct, 2) AS learning_pnl_pct,
    round(learning_exit_pct - structural_exit_pct, 2) AS shadow_edge_pp,
    round(peak_pnl_pct, 2) AS peak_pnl_pct,
    round(hold_seconds, 1) AS hold_seconds,
    CASE WHEN pnl_pct > 0 THEN 1 ELSE 0 END AS win
FROM raw_positions
ORDER BY timestamp DESC;
"""

df_trades = con.sql(trades_sql).pl()
df_trades.head(10)

IOException: IO Error: Could not connect to server error for HTTP GET to 'http://s3.seaweed.home.arpa/symm/?encoding-type=url&list-type=2&prefix=virtual%2Fclosed%2F'

In [ ]:
# Cumulative PnL Curve: Structural vs. Learning
fig, ax = plt.subplots(figsize=(12, 5))

df_pd = df_trades.reverse().to_pandas()
df_pd['cum_structural'] = df_pd['structural_pnl_pct'].cumsum()
df_pd['cum_learning'] = df_pd['learning_pnl_pct'].cumsum()

plt.plot(df_pd['cum_structural'], label='Structural (Active)', color='#4ade80', lw=2)
plt.plot(df_pd['cum_learning'], label='Learning (Shadow)', color='#38bdf8', lw=2, linestyle='--')
plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
plt.title("Cumulative PnL (%) — Structural vs Learning Stop")
plt.xlabel("Trade Count")
plt.ylabel("Cumulative Return (%)")
plt.legend()
plt.grid(True, alpha=0.2)
plt.show()

In [ ]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "iframe"

# Interactive scatter plot with Plotly
fig = px.scatter(
    df_pd, 
    x="peak_pnl_pct", 
    y="structural_pnl_pct",
    color="structural_pnl_pct",
    color_continuous_scale=["#f87171", "#fbbf24", "#4ade80"],
    hover_data=["symbol", "hold_seconds", "exit_trigger"],
    labels={"peak_pnl_pct": "Peak Profit Reached (%)", "structural_pnl_pct": "Actual Banked Profit (%)"},
    title="Peak Potential vs. Realized Banked Return"
)

# Add reference line (100% efficiency)
max_val = max(df_pd['peak_pnl_pct'].max(), 5.0)
fig.add_shape(type="line", x0=0, y0=0, x1=max_val, y1=max_val, line=dict(color="gray", dash="dot"))

fig.update_layout(template="plotly_dark", height=500)
fig.show()

In [ ]:
# Clean up exit trigger names for visual display
def clean_trigger(t):
    if not t or str(t) == 'None': return "Other"
    t = str(t).lower()
    if "structural floor" in t: return "Structural Floor"
    if "structural exhaustion" in t: return "Retest Exhaustion"
    if "invalidation" in t: return "Invalidation Cut"
    if "policy exit" in t: return "Grammar Exit"
    return "Other"

df_pd['trigger_clean'] = df_pd['exit_trigger'].apply(clean_trigger)

fig = px.pie(
    df_pd, 
    names="trigger_clean", 
    title="Distribution of Exit Triggers",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig.update_layout(template="plotly_dark", height=450)
fig.show()

In [ ]:
fig = px.scatter(
    df_pd,
    x="hold_seconds",
    y="structural_pnl_pct",
    size=df_pd['peak_pnl_pct'].clip(lower=1.0),
    color=df_pd['structural_pnl_pct'] > 0,
    color_discrete_map={True: "#4ade80", False: "#f87171"},
    hover_data=["symbol", "entry_path", "exit_trigger"],
    labels={"hold_seconds": "Hold Duration (Seconds)", "structural_pnl_pct": "Realized Return (%)"},
    title="Hold Time vs. Realized Return (Bubble Size = Peak Reach)"
)
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_dark", height=500)
fig.show()

In [ ]:
# Group by entry pattern
path_stats = df_pd.groupby('entry_path').agg(
    trades=('structural_pnl_pct', 'count'),
    total_pnl=('structural_pnl_pct', 'sum'),
    avg_pnl=('structural_pnl_pct', 'mean')
).reset_index()

# Filter out paths with only 1 trade so we see repeated patterns
path_stats = path_stats[path_stats['trades'] >= 1].sort_values(by='total_pnl', ascending=True)

fig = px.bar(
    path_stats,
    x="total_pnl",
    y="entry_path",
    orientation='h',
    color="total_pnl",
    color_continuous_scale=["#f87171", "#fbbf24", "#4ade80"],
    text=path_stats['trades'].apply(lambda x: f"{x} trades"),
    labels={"total_pnl": "Total Cumulative Return (%)", "entry_path": "Regime Pattern"},
    title="Regime Grammar Leaderboard (Which Patterns Win?)"
)
fig.add_vline(x=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_dark", height=500)
fig.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set dark theme styling to match your SYMM terminal vibe
plt.style.use('dark_background')
fig, axs = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("SYMM Strategy Diagnostics", fontsize=18, fontweight='bold', color='#f3f4f6', y=0.98)

# -------------------------------------------------------------
# 1. Top-Left: Greed vs. Lock-in (Peak vs Realized Exit)
# -------------------------------------------------------------
ax1 = axs[0, 0]
colors = ['#4ade80' if p > 0 else '#f87171' for p in df_pd['structural_pnl_pct']]
ax1.scatter(df_pd['peak_pnl_pct'], df_pd['structural_pnl_pct'], c=colors, s=80, alpha=0.85, edgecolors='white', linewidth=0.5)

# Diagonal 1:1 reference line (perfect exit efficiency)
max_peak = max(df_pd['peak_pnl_pct'].max(), 5.0)
ax1.plot([0, max_peak], [0, max_peak], color='gray', linestyle=':', label='100% Efficiency Line')
ax1.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax1.set_title("Peak Potential vs. Banked Realized Return", fontsize=13, color='#e5e7eb', pad=10)
ax1.set_xlabel("Peak Profit Reached (%)", fontsize=11)
ax1.set_ylabel("Actual Banked Return (%)", fontsize=11)
ax1.grid(True, alpha=0.15)
ax1.legend(loc='upper left')

# -------------------------------------------------------------
# 2. Top-Right: Why Did We Exit? (Exit Triggers)
# -------------------------------------------------------------
ax2 = axs[0, 1]
def clean_trigger(t):
    if not t or str(t) == 'None': return "Other"
    t = str(t).lower()
    if "structural floor" in t: return "Milestone Floor"
    if "structural exhaustion" in t: return "Retest Exhaustion"
    if "invalidation" in t: return "Invalidation Cut"
    return "Other"

trigger_counts = df_pd['exit_trigger'].apply(clean_trigger).value_counts()
palette = ['#38bdf8', '#fbbf24', '#f87171', '#a78bfa']
bars = ax2.bar(trigger_counts.index, trigger_counts.values, color=palette[:len(trigger_counts)], alpha=0.85)
ax2.set_title("Exit Trigger Breakdown", fontsize=13, color='#e5e7eb', pad=10)
ax2.set_ylabel("Trade Count", fontsize=11)
ax2.grid(True, axis='y', alpha=0.15)
for bar in bars:
    height = bar.get_height()
    ax2.annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', color='#f3f4f6')

# -------------------------------------------------------------
# 3. Bottom-Left: Patience Curve (Hold Duration vs Return)
# -------------------------------------------------------------
ax3 = axs[1, 0]
ax3.scatter(df_pd['hold_seconds'] / 60.0, df_pd['structural_pnl_pct'], c=colors, s=80, alpha=0.85, edgecolors='white', linewidth=0.5)
ax3.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax3.set_title("Hold Time vs. Banked Return", fontsize=13, color='#e5e7eb', pad=10)
ax3.set_xlabel("Hold Duration (Minutes)", fontsize=11)
ax3.set_ylabel("Actual Banked Return (%)", fontsize=11)
ax3.grid(True, alpha=0.15)

# -------------------------------------------------------------
# 4. Bottom-Right: Radix Grammar Leaderboard
# -------------------------------------------------------------
ax4 = axs[1, 1]
path_stats = df_pd.groupby('entry_path')['structural_pnl_pct'].sum().sort_values()
bar_colors = ['#4ade80' if val > 0 else '#f87171' for val in path_stats.values]
ax4.barh(path_stats.index, path_stats.values, color=bar_colors, alpha=0.85)
ax4.axvline(0, color='gray', linewidth=0.8, linestyle='--')
ax4.set_title("Total Cumulative Return by Grammar Path", fontsize=13, color='#e5e7eb', pad=10)
ax4.set_xlabel("Cumulative PnL (%)", fontsize=11)
ax4.grid(True, axis='x', alpha=0.15)

plt.tight_layout()
plt.show()

In [ ]:
l3_sql = """
WITH l3_frames AS (
    SELECT 
        channel, 
        type AS frame_type,
        data
    FROM read_json(
        's3://symm/market/level3/**/*.ndjson', 
        format = 'newline_delimited',
        columns = {'channel': 'VARCHAR', 'type': 'VARCHAR', 'data': 'JSON'}
    )
),
l3_events AS (
    SELECT 
        f.channel,
        f.frame_type,
        json_extract_string(e, '$.symbol') AS symbol,
        json_extract_string(e, '$.type') AS event_type,
        TRY_CAST(json_extract_string(e, '$.timestamp') AS TIMESTAMP) AS event_time,
        TRY_CAST(json_extract(e, '$.checksum') AS UBIGINT) AS checksum,
        json_extract(e, '$.bids')::VARCHAR AS bids,
        json_extract(e, '$.asks')::VARCHAR AS asks
    FROM l3_frames f, UNNEST(CAST(f.data AS JSON[])) AS t(e)
)
SELECT 
    symbol,
    frame_type,
    event_type,
    event_time,
    checksum,
    bids,
    asks
FROM l3_events
WHERE symbol = 'WINGS/USD'   -- try 'WINGS/USD' or 'Q/USD'
ORDER BY event_time DESC
LIMIT 50;
"""

df_l3 = con.sql(l3_sql).df()  # or .pl() for Polars
df_l3.head(10)

In [ ]:
import json
import pandas as pd
import plotly.graph_objects as go

# 1. Grab just ONE single part file (lightning fast < 1 second!)
sample_file = con.sql("SELECT file FROM glob('s3://symm/market/level3/**/*.ndjson') LIMIT 1").df()['file'][0]
print("Inspecting single L3 tape file:", sample_file)

# 2. Pull a snapshot from that file
snapshot_df = con.sql(f"""
    SELECT 
        f.type AS frame_type,
        json_extract_string(e, '$.symbol') AS symbol,
        e->'$.bids' AS bids,
        e->'$.asks' AS asks
    FROM read_json('{sample_file}', format='newline_delimited') f,
         UNNEST(CAST(f.data AS JSON[])) AS t(e)
    WHERE f.type = 'snapshot'
    LIMIT 1;
""").df()

selected_symbol = snapshot_df['symbol'].iloc[0]
raw_bids = json.loads(snapshot_df['bids'].iloc[0])
raw_asks = json.loads(snapshot_df['asks'].iloc[0])

# 3. Process bids and asks into cumulative dollar depth
bids_df = pd.DataFrame(raw_bids)
bids_df['price'] = bids_df['limit_price'].astype(float)
bids_df['qty'] = bids_df['order_qty'].astype(float)
bids_df = bids_df.sort_values(by='price', ascending=False)
bids_df['cum_cash'] = (bids_df['price'] * bids_df['qty']).cumsum()

asks_df = pd.DataFrame(raw_asks)
asks_df['price'] = asks_df['limit_price'].astype(float)
asks_df['qty'] = asks_df['order_qty'].astype(float)
asks_df = asks_df.sort_values(by='price', ascending=True)
asks_df['cum_cash'] = (asks_df['price'] * asks_df['qty']).cumsum()

# 4. Plot the Order Book Depth Canyon
fig = go.Figure()

# Bids (Green fill)
fig.add_trace(go.Scatter(
    x=bids_df['price'], 
    y=bids_df['cum_cash'], 
    fill='tozeroy', 
    name='Bid Support ($)', 
    line=dict(color='#4ade80', width=2)
))

# Asks (Red fill)
fig.add_trace(go.Scatter(
    x=asks_df['price'], 
    y=asks_df['cum_cash'], 
    fill='tozeroy', 
    name='Ask Resistance ($)', 
    line=dict(color='#f87171', width=2)
))

best_bid = bids_df['price'].iloc[0]
best_ask = asks_df['price'].iloc[0]
spread_pct = ((best_ask - best_bid) / best_bid) * 100

fig.update_layout(
    template="plotly_dark",
    title=f"Level 3 Order Book Depth Canyon — {selected_symbol} (Spread: {spread_pct:.2f}%)",
    xaxis_title="Price ($)",
    yaxis_title="Cumulative Depth ($ Cash)",
    height=550,
    hovermode="x unified"
)
fig.show()

In [ ]:
# Extract all L3 delta events from that same part file
events_df = con.sql(f"""
    WITH raw_orders AS (
        SELECT 
            json_extract_string(e, '$.symbol') AS symbol,
            'bid' AS side,
            UNNEST(CAST(e->'$.bids' AS JSON[])) AS o
        FROM read_json('{sample_file}', format='newline_delimited') f,
             UNNEST(CAST(f.data AS JSON[])) AS t(e)
        WHERE f.type != 'snapshot'
        
        UNION ALL
        
        SELECT 
            json_extract_string(e, '$.symbol') AS symbol,
            'ask' AS side,
            UNNEST(CAST(e->'$.asks' AS JSON[])) AS o
        FROM read_json('{sample_file}', format='newline_delimited') f,
             UNNEST(CAST(f.data AS JSON[])) AS t(e)
        WHERE f.type != 'snapshot'
    )
    SELECT 
        symbol,
        side,
        COALESCE(json_extract_string(o, '$.event'), 'add') AS event,
        count(*) AS event_count,
        round(sum(TRY_CAST(json_extract(o, '$.order_qty') AS DOUBLE)), 2) AS total_qty
    FROM raw_orders
    GROUP BY symbol, side, event
    ORDER BY event_count DESC;
""").df()

# Plot the top symbols' microstructural activity
top_symbols = events_df.groupby('symbol')['event_count'].sum().nlargest(6).index
sub_df = events_df[events_df['symbol'].isin(top_symbols)]
sub_df['label'] = sub_df['side'].str.upper() + " " + sub_df['event'].str.upper()

fig = px.bar(
    sub_df,
    x="symbol",
    y="event_count",
    color="label",
    barmode="group",
    title="Level 3 Market Maker Churn: Adds vs. Cancels by Side",
    labels={"event_count": "Event Count", "symbol": "Pair"},
    color_discrete_map={
        "BID ADD": "#4ade80",
        "BID DELETE": "#15803d",
        "ASK ADD": "#f87171",
        "ASK DELETE": "#991b1b"
    }
)
fig.update_layout(template="plotly_dark", height=500)
fig.show()

In [8]:
import socket

host = "s3.seaweed.home.arpa"

try:
    ip = socket.gethostbyname(host)
    print(f"✓ Resolved {host} -> {ip}")
    
    # Test ports (80 for standard HTTP, 8333 for default SeaweedFS S3)
    for port in [80, 8333, 8888, 9000]:
        s = socket.socket()
        s.settimeout(1.5)
        if s.connect_ex((ip, port)) == 0:
            print(f"✓ Port {port} is OPEN!")
        s.close()
except socket.gaierror as e:
    print(f"✗ DNS Resolution failed for {host}: {e}")

✓ Resolved s3.seaweed.home.arpa -> 192.168.0.240
